# 轻量级 Transformer 机器翻译 Demo\n\n本 Notebook 展示如何加载在 WMT 上训练好的 Transformer 模型，\n并对中文句子进行翻译，同时可视化多头注意力权重。

In [ ]:
# 初始化环境与加载依赖\nimport os\nimport yaml\nimport torch\nimport sentencepiece as spm\nimport matplotlib.pyplot as plt\n\nfrom transformer.model import Transformer\nfrom transformer.utils import load_checkpoint, set_seed\n\nbase_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))\nconfig_path = os.path.join(base_dir, "transformer", "config.yaml")\nckpt_path = os.path.join(base_dir, "checkpoints", "step_50000.pt")  # 根据实际路径修改\n\nwith open(config_path, "r", encoding="utf-8") as f:\n    cfg = yaml.safe_load(f)\n\ndata_cfg = cfg["data"]\nsp_model_path = os.path.join(data_cfg["data_dir"], f"spm_{data_cfg['vocab_size']}.model")\n\ndevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nset_seed(cfg.get("seed", 42))\n\nsp = spm.SentencePieceProcessor()\nsp.load(sp_model_path)\nvocab_size = sp.get_vocab_size()\n\nmcfg = cfg["model"]\nmodel = Transformer(\n    src_vocab_size=vocab_size,\n    tgt_vocab_size=vocab_size,\n    d_model=mcfg["d_model"],\n    n_layers=mcfg["n_layers"],\n    n_heads=mcfg["n_heads"],\n    d_ff=mcfg["d_ff"],\n    dropout=mcfg["dropout"],\n    share_embeddings=mcfg.get("share_embeddings", True),\n).to(device)\n\nload_checkpoint(ckpt_path, model, optimizer=None, scheduler=None, map_location=device)\nmodel.eval()\nprint("模型与 SentencePiece 加载完成，设备:", device)

In [ ]:
# 输入中文句子并生成翻译，返回译文及注意力权重\nfrom typing import List\n\ndef translate_and_get_attn(text: str, max_len: int = 64):\n    pad_id = sp.pad_id()\n    bos_id = sp.bos_id()\n    eos_id = sp.eos_id()\n\n    ids = [bos_id] + sp.encode(text, out_type=int) + [eos_id]\n    src = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)\n    src_mask = src.ne(pad_id).unsqueeze(1).unsqueeze(2)\n\n    # 保存注意力的 hook 容器\n    attn_maps: List[torch.Tensor] = []\n\n    def hook_attn(module, inp, out):\n        _, attn = out\n        attn_maps.append(attn.detach().cpu())\n\n    handles = []\n    for layer in model.encoder.layers:\n        handles.append(layer.self_attn.register_forward_hook(hook_attn))\n\n    ys = torch.full((1, 1), bos_id, dtype=torch.long, device=device)\n    with torch.no_grad():\n        memory = model.encode(src, src_mask)\n        for _ in range(max_len - 1):\n            tgt_mask = ys.ne(pad_id).unsqueeze(1).unsqueeze(2)\n            tgt_sub = torch.tril(\n                torch.ones(1, ys.size(1), ys.size(1), device=device, dtype=torch.bool)\n            )\n            tgt_mask = tgt_mask & tgt_sub.unsqueeze(1)\n            out = model.decode(ys, memory, tgt_mask, src_mask)\n            logits = model.generator(out[:, -1])\n            next_token = logits.log_softmax(dim=-1).argmax(dim=-1)\n            ys = torch.cat([ys, next_token.unsqueeze(1)], dim=1)\n            if next_token.item() == eos_id:\n                break\n\n    for h in handles:\n        h.remove()\n\n    hyp_ids = ys[0].tolist()[1:]\n    if eos_id in hyp_ids:\n        hyp_ids = hyp_ids[: hyp_ids.index(eos_id)]\n    hyp = sp.decode(hyp_ids)\n    return hyp, attn_maps\n\nsrc_text = "这是一个简单的例子。"\nhyp_text, attn_maps = translate_and_get_attn(src_text)\nprint("源:", src_text)\nprint("译:", hyp_text)

In [ ]:
# 可视化第一层多头注意力，这里绘制 Encoder 第 1 层平均注意力\ndef plot_attn(attn):\n    # attn 形状: (batch, heads, seq, seq)\n    attn_mean = attn[0].mean(dim=0)  # (seq, seq)\n    plt.figure(figsize=(6, 5))\n    plt.imshow(attn_mean, cmap="viridis")\n    plt.colorbar()\n    plt.title("Encoder Layer 1 平均注意力")\n    plt.xlabel("Key 位置")\n    plt.ylabel("Query 位置")\n    plt.show()\n\nif attn_maps:\n    plot_attn(attn_maps[0])\nelse:\n    print("注意力权重为空，请先成功运行上一单元格。")